In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)

# =====================================================
# CONFIG
# =====================================================

N_ROWS = 100_000
NUM_DRIVERS = 500

# =====================================================
# DRIVER PROFILES
# =====================================================

drivers = []

for i in range(NUM_DRIVERS):

    persona = np.random.choice(
        [
            "aggressive",
            "balanced",
            "lazy",
            "backhaul_hunter"
        ],
        p=[0.25, 0.45, 0.15, 0.15]
    )

    historical_accept_rate = np.clip(
        np.random.normal(0.65, 0.15),
        0.15,
        0.98
    )

    historical_cancel_rate = np.clip(
        np.random.beta(2, 15),
        0,
        0.35
    )

    preferred_framework = np.random.choice(
        [1, 2, 3, 4, 5],
        p=[0.22, 0.28, 0.15, 0.20, 0.15]
    )

    drivers.append({
        "driver_id": f"D{i:04d}",
        "persona": persona,
        "historical_accept_rate": historical_accept_rate,
        "historical_cancel_rate": historical_cancel_rate,
        "historical_framework_preference": preferred_framework
    })

# =====================================================
# MAIN GENERATION
# =====================================================

rows = []

for _ in range(N_ROWS):

    drv = drivers[np.random.randint(NUM_DRIVERS)]

    # =================================================
    # TIME CONTEXT
    # =================================================

    hour_of_day = np.random.randint(0, 24)

    is_peak_hour = int(
        (7 <= hour_of_day <= 9)
        or
        (16 <= hour_of_day <= 20)
    )

    rain_flag = int(np.random.rand() < 0.12)

    # =================================================
    # DRIVER LIVE STATE
    # =================================================

    fatigue_level = np.clip(
        np.random.gamma(2, 2),
        0,
        14
    )

    earnings_today = np.random.gamma(
        2.5,
        900
    )

    idle_time_minutes = np.random.gamma(
        2,
        25
    )

    has_rejected_this_job = int(
        np.random.rand() < 0.08
    )

    # =================================================
    # JOB
    # =================================================

    job_distance = np.random.gamma(
        2.5,
        15
    )

    price_per_km = np.random.uniform(
        12,
        45
    )

    if rain_flag:
        price_per_km *= np.random.uniform(1.05, 1.20)

    if is_peak_hour:
        price_per_km *= np.random.uniform(1.05, 1.15)

    job_price = job_distance * price_per_km

    # =================================================
    # FRAMEWORK
    # =================================================

    matched_framework = np.random.choice(
        [1, 2, 3, 4, 5],
        p=[0.22, 0.28, 0.15, 0.20, 0.15]
    )

    framework_match = int(
        matched_framework ==
        drv["historical_framework_preference"]
    )

    # =================================================
    # ROUTE STATE
    # =================================================

    route_completion_ratio = np.clip(
        np.random.beta(2, 2),
        0,
        1
    )

    remaining_stops = max(
        0,
        int(
            np.random.poisson(
                5 * (1 - route_completion_ratio)
            )
        )
    )

    is_returning = int(
        route_completion_ratio > 0.85
        and
        remaining_stops <= 1
    )

    current_load_utilization = np.clip(
        np.random.beta(2, 2),
        0,
        1
    )

    distance_to_home_depot = np.random.gamma(
        2,
        15
    )

    pickup_detour_km = np.random.gamma(
        2,
        3
    )

    total_detour_km = (
        pickup_detour_km
        +
        np.random.gamma(2, 4)
    )

    eta_delay_minutes = np.clip(
        total_detour_km
        *
        np.random.uniform(2, 4),
        0,
        120
    )

    # =================================================
    # ECONOMICS
    # =================================================

    estimated_fuel_cost = (
        total_detour_km
        *
        np.random.uniform(2.5, 4.5)
    )

    expected_margin = (
        job_price
        -
        estimated_fuel_cost
    )

    margin_per_km = (
        expected_margin
        /
        max(job_distance, 1)
    )

    # =================================================
    # MARKETPLACE
    # =================================================

    nearby_driver_count = np.random.poisson(
        15
    )

    marketplace_job_density = np.random.poisson(
        12
    )

    supply_demand_ratio = (
        nearby_driver_count
        /
        max(marketplace_job_density, 1)
    )

    # =================================================
    # DRIVER PERSONA UTILITY
    # =================================================

    utility = -1.2

    utility += (
        drv["historical_accept_rate"]
        * 1.4
    )

    utility -= (
        drv["historical_cancel_rate"]
        * 2.0
    )

    utility += (
        np.tanh(price_per_km / 25)
        * 1.2
    )

    utility += (
        expected_margin / 1200
    )

    utility -= (
        pickup_detour_km / 20
    )

    utility -= (
        eta_delay_minutes / 90
    )

    utility -= (
        fatigue_level / 12
    )

    utility += (
        np.tanh(idle_time_minutes / 30)
        * 0.7
    )

    utility -= (
        np.tanh(earnings_today / 5000)
        * 0.4
    )

    utility += (
        framework_match
        * 0.7
    )

    utility += (
        route_completion_ratio
        * 0.3
    )

    utility += (
        is_returning
        * 0.5
    )

    utility -= (
        current_load_utilization
        * 0.8
    )

    utility -= (
        has_rejected_this_job
        * 1.5
    )

    utility -= (
        supply_demand_ratio
        * 0.25
    )

    if rain_flag:
        utility -= 0.15

    # =================================================
    # PERSONA EFFECTS
    # =================================================

    if drv["persona"] == "aggressive":

        utility += (
            expected_margin / 1000
        )

        utility -= (
            fatigue_level / 20
        )

    elif drv["persona"] == "lazy":

        utility -= (
            pickup_detour_km / 12
        )

        utility -= (
            fatigue_level / 6
        )

    elif drv["persona"] == "backhaul_hunter":

        if is_returning:
            utility += 1.0

        if framework_match:
            utility += 0.8

    utility += np.random.normal(
        0,
        0.35
    )

    # =================================================
    # LABEL
    # =================================================

    accept_probability = (
        1 /
        (1 + np.exp(-utility))
    )

    accept_probability = np.clip(
        accept_probability,
        0.01,
        0.99
    )

    accepted = int(
        np.random.rand()
        <
        accept_probability
    )

    # =================================================
    # SAVE
    # =================================================

    rows.append({

        "driver_id":
            drv["driver_id"],

        "historical_accept_rate":
            round(
                drv["historical_accept_rate"],
                4
            ),

        "historical_cancel_rate":
            round(
                drv["historical_cancel_rate"],
                4
            ),

        "historical_framework_preference":
            drv["historical_framework_preference"],

        "matched_framework":
            matched_framework,

        "framework_match":
            framework_match,

        "has_rejected_this_job":
            has_rejected_this_job,

        "fatigue_level":
            round(fatigue_level, 2),

        "earnings_today":
            round(earnings_today, 2),

        "idle_time_minutes":
            round(idle_time_minutes, 2),

        "hour_of_day":
            hour_of_day,

        "current_load_utilization":
            round(current_load_utilization, 4),

        "route_completion_ratio":
            round(route_completion_ratio, 4),

        "is_returning":
            is_returning,

        "distance_to_home_depot":
            round(distance_to_home_depot, 2),

        "pickup_detour_km":
            round(pickup_detour_km, 2),

        "total_detour_km":
            round(total_detour_km, 2),

        "eta_delay_minutes":
            round(eta_delay_minutes, 2),

        "estimated_fuel_cost":
            round(estimated_fuel_cost, 2),

        "expected_margin":
            round(expected_margin, 2),

        "margin_per_km":
            round(margin_per_km, 2),

        "nearby_driver_count":
            nearby_driver_count,

        "marketplace_job_density":
            marketplace_job_density,

        "supply_demand_ratio":
            round(
                supply_demand_ratio,
                3
            ),

        "accepted":
            accepted
    })

# =====================================================
# DATAFRAME
# =====================================================

df = pd.DataFrame(rows)

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print("Shape:", df.shape)
print("Accept Rate:", round(df["accepted"].mean(), 4))
print()

print("Null Values:")
print(df.isnull().sum().sum())
print()

print("Accepted Distribution:")
print(df["accepted"].value_counts(normalize=True))
print()

print("Framework Match Acceptance:")
print(
    df.groupby("framework_match")["accepted"]
      .mean()
      .round(4)
)
print()

print("Correlation Top Features:")
corr = (
    df.select_dtypes(include=[np.number])
      .corr()["accepted"]
      .drop("accepted")
      .abs()
      .sort_values(ascending=False)
)

print(corr.head(15))

# =====================================================
# EXPORT
# =====================================================

output_path = "matching_dataset_final.csv"

df.to_csv(
    output_path,
    index=False
)

print()
print(f"Exported -> {output_path}")
print()

print("Columns:")
print(df.columns.tolist())

DATASET SUMMARY
Shape: (100000, 25)
Accept Rate: 0.5156

Null Values:
0

Accepted Distribution:
accepted
1    0.5156
0    0.4844
Name: proportion, dtype: float64

Framework Match Acceptance:
framework_match
0    0.4852
1    0.6271
Name: accepted, dtype: float64

Correlation Top Features:
expected_margin             0.309842
margin_per_km               0.233891
eta_delay_minutes           0.144687
pickup_detour_km            0.143147
total_detour_km             0.140987
has_rejected_this_job       0.135419
estimated_fuel_cost         0.131973
framework_match             0.116446
fatigue_level               0.115523
current_load_utilization    0.066776
historical_accept_rate      0.065745
is_returning                0.063705
supply_demand_ratio         0.052558
route_completion_ratio      0.048736
idle_time_minutes           0.045325
Name: accepted, dtype: float64

Exported -> matching_dataset_final.csv

Columns:
['driver_id', 'historical_accept_rate', 'historical_cancel_rate', 'historic